# Logistic Regression (Model -1)

In [12]:
# MODEL 1 - Logistic Regression with TF-IDF features
# TF-IDF converts text to numbers based on word importance
# Logistic Regression then classifies which answer (A-E) is correct

# ==================== creating TF-IDF features =======================

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf    = tfidf_vectorizer.fit_transform(train['full_text'])
X_test_tfidf     = tfidf_vectorizer.transform(test['full_text'])

print('TF-IDF feature matrix shape:', X_train_tfidf.shape)

# ============ starting wandb run for logistic regression ==================

run1 = wandb.init(
    project = '23f3001514-t22026',
    entity  = '23f3001514-instituition',
    name    = 'logistic-regression-tfidf',
    config  = {
        'model'        : 'LogisticRegression',
        'features'     : 'TF-IDF',
        'max_features' : 5000,
        'C'            : 1.0,
        'max_iter'     : 1000
    }
)

# ============== training logistic regression =================

lr_model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', multi_class='multinomial')
lr_model.fit(X_train_tfidf, y_train)

# ============ calculating cross validation scores ==================

lr_f1_scores  = cross_val_score(lr_model, X_train_tfidf, y_train, cv=5, scoring='f1_macro')
lr_acc_scores = cross_val_score(lr_model, X_train_tfidf, y_train, cv=5, scoring='accuracy')

# =========== calculating mean score of metrics ===============

lr_f1  = lr_f1_scores.mean()
lr_acc = lr_acc_scores.mean()

# ============= calculating mAP@3 on training set ===============

lr_train_probs = lr_model.predict_proba(X_train_tfidf)
lr_train_top3  = []
for i in range(len(lr_train_probs)):
    top3 = get_top3_from_probs(lr_train_probs[i], list(lr_model.classes_))
    lr_train_top3.append(top3)

lr_map3 = map_at_k(y_train.tolist(), lr_train_top3)

print('Logistic Regression Results:')
print('  CV Macro F1  :', round(lr_f1, 4))
print('  CV Accuracy  :', round(lr_acc, 4))
print('  Train mAP@3  :', round(lr_map3, 4))

# =========== logging to wandb ==================

wandb.log({
    'macro_f1' : lr_f1,
    'accuracy' : lr_acc,
    'map_at_3' : lr_map3
})
run1.finish()

# =========== getting test predictions =================

lr_test_probs = lr_model.predict_proba(X_test_tfidf)
print('='*50)
print('LR done!')

TF-IDF feature matrix shape: (2000, 2940)


Logistic Regression Results:
  CV Macro F1  : 0.9987
  CV Accuracy  : 0.9985
  Train mAP@3  : 1.0


accuracy,▁
macro_f1,▁
map_at_3,▁
accuracy,0.9985
macro_f1,0.99873
map_at_3,1


LR done!


**INSIGHT:**

Logistic Regression Insights:

1. TF-IDF created only 2940 features out of max 5000.
   This means the dataset vocabulary is small and domain specific.

2. CV Accuracy of 99.85% and Macro F1 of 99.87% seem very high.
   This suggests the model is learning a strong lexical overlap pattern —
   the correct answer tends to repeat key words from the question prompt.
   TF-IDF captures this overlap very effectively.

3. Train mAP@3 of 1.0 means for every question the correct answer
   appeared in the top 3 predictions. Perfect ranking on training data.


**INSIGHT:**

1. Vocabulary size is only 2975 words — much smaller than our
   VOCAB_SIZE_MAX of 15000. This confirms the dataset is domain
   specific with limited unique terminology across all questions.

2. MAX_LENGTH of 256 was chosen based on EDA — most questions
   with all 5 options combined are under 256 words. Setting it
   higher would waste memory with unnecessary padding.

3. Words appearing only once are excluded (count >= 2).
   Single occurrence words are likely typos or rare terms
   that don't help the model learn general patterns.

4. Train/val split of 80/20 gives 1600 training samples and
   400 validation samples. Validation data is never used for
   training — only for checking generalization during training.

5. Batch size of 32 is a good balance between memory efficiency
   and training stability. Too large = GPU memory error.
   Too small = noisy gradient updates.

# TEXTCNN (Model -2)

In [21]:
# MODEL 2 - TextCNN (built completely from scratch)
# CNN for text classification works by sliding filters over the word embeddings
# different filter sizes capture different length patterns (bigrams, trigrams etc)
# i am using kernel sizes 2, 3, 4 to capture 2-word, 3-word and 4-word patterns

class TextCNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_classes, num_filters, kernel_sizes):
        super(TextCNN, self).__init__()

        # ======= embedding layer converts word indices to dense vectors =========
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        #  ========== convolutional layers with different kernel sizes ===========
        # ========== each one captures patterns of different lengths ============
        
        self.conv2 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=2)
        self.conv3 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=3)
        self.conv4 = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=4)

        # ====== dropout for regularization - prevents overfitting ========
        
        self.dropout = nn.Dropout(p=0.5)

        #  ====== final linear layer : maps to number of classes (5 options) ===========
        #  =========== 3 * num_filters because we have 3 different kernel sizes =============
        
        self.fc = nn.Linear(3 * num_filters, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length)

        # ======= getting embeddings ========
        
        embedded = self.embedding(x)
        # ===== shape: (batch_size, sequence_length, embedding_dim) =========

        # ======= conv1d expects (batch_size, channels, length) so we permute the dimensions ==========
     
        embedded = embedded.permute(0, 2, 1) # shape: (batch_size, embedding_dim, sequence_length)

        # ========== applying each convolutional filter and relu activation ===============
        
        out2 = torch.relu(self.conv2(embedded))
        out3 = torch.relu(self.conv3(embedded))
        out4 = torch.relu(self.conv4(embedded))

        # ========= global max pooling : keeps only the highest value from each filter =============
        
        pool2 = torch.max(out2, dim=2)[0]
        pool3 = torch.max(out3, dim=2)[0]
        pool4 = torch.max(out4, dim=2)[0]

        # ======= concatenating all pooled outputs in a single long vector ===========
        
        concatenated = torch.cat([pool2, pool3, pool4], dim=1)

        # ===== applying dropout =======
        
        dropped = self.dropout(concatenated)

        #  ======= final classification =======
        
        output = self.fc(dropped)

        return output


# ================ creating the TextCNN model ====================

cnn_model = TextCNN(
    vocab_size    = VOCAB_SIZE,
    embedding_dim = 128,
    num_classes   = 5,
    num_filters   = 128,
    kernel_sizes  = [2, 3, 4]
)
cnn_model = cnn_model.to(device)

total_params = sum(p.numel() for p in cnn_model.parameters())
print('TextCNN total parameters:', total_params)
print('='*50)
print()
print(cnn_model)

TextCNN total parameters: 530565

TextCNN(
  (embedding): Embedding(2975, 128, padding_idx=0)
  (conv2): Conv1d(128, 128, kernel_size=(2,), stride=(1,))
  (conv3): Conv1d(128, 128, kernel_size=(3,), stride=(1,))
  (conv4): Conv1d(128, 128, kernel_size=(4,), stride=(1,))
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=384, out_features=5, bias=True)
)


**INSIGHT :**

1. Total parameters: 530,565
   Most parameters are in the embedding layer (380,800)
   which learns word representations during training.

2. Three separate conv filters capture patterns of different lengths.
   kernel_size=2 catches short patterns like "not correct"
   kernel_size=3 catches medium patterns like "is the answer"
   kernel_size=4 catches longer contextual patterns.

3. Global max pooling reduces each filter output to a single value
   — the most important feature found anywhere in the text.
   This makes the model position-invariant — it doesn't matter
   where in the text the pattern appears.

4. Dropout of 0.5 randomly disables half the neurons during training.
   This forces the model to learn redundant representations
   and prevents it from memorizing training examples.

5. Final layer maps 384 features → 5 scores.
   The class with highest score = predicted answer.

# TEXTCNN Training Loop

In [22]:
# ========= training the TextCNN model =============

run2 = wandb.init(
    project = '23f3001514-t22026',
    entity  = '23f3001514-instituition',
    name    = 'textcnn-scratch',
    config  = {
        'model'        : 'TextCNN',
        'vocab_size'   : VOCAB_SIZE,
        'embedding_dim': 128,
        'num_filters'  : 128,
        'kernel_sizes' : [2, 3, 4],
        'dropout'      : 0.5,
        'batch_size'   : 32,
        'learning_rate': 0.001,
        'epochs'       : 20
    }
)

# ========= loss function and optimizer =============

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001, weight_decay=1e-4)

# ============ learning rate scheduler : reduces lr when validation doesnt improve ===============

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

best_val_accuracy = 0
patience_counter  = 0
NUM_EPOCHS        = 20
PATIENCE          = 5

for epoch in range(NUM_EPOCHS):

    # ====== training phase =======
    
    cnn_model.train()
    train_loss    = 0
    train_correct = 0
    train_total   = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # ====== forward pass ========
        
        optimizer.zero_grad()
        outputs = cnn_model(batch_x)
        loss    = criterion(outputs, batch_y)

        #  ====== backward pass (backpropagation) ==========
        loss.backward()

        # ======= gradient clipping to prevent exploding gradients ========
        nn.utils.clip_grad_norm_(cnn_model.parameters(), max_norm=1.0)

        optimizer.step()

        train_loss    += loss.item()
        predictions    = outputs.argmax(dim=1)
        train_correct += (predictions == batch_y).sum().item()
        train_total   += batch_y.size(0)

    train_acc = train_correct / train_total

    # ====== validation phase ======
    cnn_model.eval()
    val_correct   = 0
    val_total     = 0
    all_val_preds = []
    all_val_true  = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x  = batch_x.to(device)
            batch_y  = batch_y.to(device)
            outputs  = cnn_model(batch_x)
            preds    = outputs.argmax(dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total   += batch_y.size(0)
            all_val_preds.extend(preds.cpu().numpy().tolist())
            all_val_true.extend(batch_y.cpu().numpy().tolist())

    val_acc = val_correct / val_total
    val_f1  = f1_score(all_val_true, all_val_preds, average='macro')

    scheduler.step(val_acc)

    # ======= logging to wandb ==========
    
    wandb.log({
        'epoch'        : epoch + 1,
        'train_loss'   : train_loss / len(train_loader),
        'train_acc'    : train_acc,
        'val_acc'      : val_acc,
        'val_macro_f1' : val_f1
    })

    # ======== saving best model ==============
    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        torch.save(cnn_model.state_dict(), '/kaggle/working/best_cnn.pt')
        patience_counter = 0
    else:
        patience_counter += 1

    # ======== early stopping ============
    
    if patience_counter >= PATIENCE:
        print(f'early stopping at epoch {epoch + 1}')
        break

    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1}/{NUM_EPOCHS} | train acc: {train_acc:.4f} | val acc: {val_acc:.4f} | val f1: {val_f1:.4f}')

wandb.log({'best_val_acc': best_val_accuracy})
run2.finish()

# ========= loading best weights ===============

cnn_model.load_state_dict(torch.load('/kaggle/working/best_cnn.pt'))
print(f'TextCNN training done! Best val accuracy: {best_val_accuracy:.4f}')


# ========== getting test predictions from CNN =============

cnn_model.eval()
cnn_test_probs_list = []

with torch.no_grad():
    for batch in test_loader:
        batch_x  = batch[0].to(device)
        outputs  = cnn_model(batch_x)
        probs    = torch.softmax(outputs, dim=1)
        cnn_test_probs_list.append(probs.cpu().numpy())

cnn_test_probs = np.concatenate(cnn_test_probs_list, axis=0)
print('CNN test predictions shape:', cnn_test_probs.shape)

epoch 5/20 | train acc: 0.9906 | val acc: 1.0000 | val f1: 1.0000
early stopping at epoch 7


best_val_acc,▁
epoch,▁▂▃▅▆▇█
train_acc,▁▆█████
train_loss,█▄▂▁▁▁▁
val_acc,▁██████
val_macro_f1,▁██████
best_val_acc,1
epoch,7
train_acc,0.99875
train_loss,0.01939
val_acc,1


TextCNN training done! Best val accuracy: 1.0000
CNN test predictions shape: (500, 5)


# TEXTCRNN (Model -3)

In [23]:
# MODEL 3 - TextCRNN (CNN + Bidirectional LSTM) built from scratch
# this model combines CNN for local patterns and LSTM for sequential patterns
# bidirectional LSTM reads the text both forward and backward
# this helps understand context from both directions

class TextCRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, num_classes, num_filters, hidden_size):
        super(TextCRNN, self).__init__()

        # ==== embedding layer =====
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # ======= CNN layer to extract local features =======
        self.conv = nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=3, padding=1)

        # ========== bidirectional LSTM with 2 layers =============
        # ====== bidirectional means it processes text both left to right and right to left ===========
        
        self.lstm = nn.LSTM(
            input_size    = num_filters,
            hidden_size   = hidden_size,
            num_layers    = 2,
            batch_first   = True,
            dropout       = 0.3,
            bidirectional = True
        )

        self.dropout = nn.Dropout(p=0.5)

        # ========== hidden_size * 2 because bidirectional (forward + backward) ============
        
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x shape: (batch_size, sequence_length)

        # embedding
        embedded = self.embedding(x)
        # shape: (batch_size, sequence_length, embedding_dim)

        # permute for conv1d
        embedded = embedded.permute(0, 2, 1) # shape: (batch_size, embedding_dim, sequence_length)

        # CNN to get local features
        conv_out = torch.relu(self.conv(embedded))
        # shape: (batch_size, num_filters, sequence_length)

        # ===== permute back for LSTM =====
        conv_out = conv_out.permute(0, 2, 1) # shape: (batch_size, sequence_length, num_filters)

        # ======= LSTM : we only need the final hidden state ==========
        lstm_out, (hidden, cell) = self.lstm(conv_out)

        # ====== hidden shape: (num_layers * 2, batch_size, hidden_size) ==========
        # ====== taking last layer forward and backward hidden states ============
        
        forward_hidden  = hidden[-2]  # ===== last layer forward =======
        backward_hidden = hidden[-1]  # ====== last layer backward ========

        # ====== concatenating both directions ========
        
        combined = torch.cat([forward_hidden, backward_hidden], dim=1) # shape: (batch_size, hidden_size * 2)

        dropped = self.dropout(combined)
        output  = self.fc(dropped)

        return output


# =============== creating the TextCRNN model ===============

crnn_model = TextCRNN(
    vocab_size    = VOCAB_SIZE,
    embedding_dim = 128,
    num_classes   = 5,
    num_filters   = 128,
    hidden_size   = 128
)
crnn_model = crnn_model.to(device)

total_params = sum(p.numel() for p in crnn_model.parameters())
print('TextCRNN total parameters:', total_params)
print('='*50)
print()
print(crnn_model)

TextCRNN total parameters: 1090821

TextCRNN(
  (embedding): Embedding(2975, 128, padding_idx=0)
  (conv): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=5, bias=True)
)


**INSIGHT:**

1. Total parameters: 1,090,821
   More than double TextCNN (530,565) because BiLSTM adds
   significant complexity with its gating mechanisms.

2. CNN + BiLSTM combination gives two types of understanding:
   CNN → local patterns (what words appear together)
   BiLSTM → sequential context (how meaning flows through text)

3. padding=1 in conv layer keeps sequence length unchanged.
   This is necessary because LSTM needs the complete sequence
   to understand long range dependencies.

4. bidirectional=True means two LSTM passes happen:
   Forward: reads question left to right
   Backward: reads question right to left
   Combining both gives the model full sentence context.

5. num_layers=2 means two LSTM layers are stacked.
   First layer learns basic patterns.
   Second layer learns more abstract higher level patterns.

6. Despite having 2x more parameters than TextCNN,
   TextCRNN received only 10% weight in ensemble.
   This is because with only 2000 training samples,
   the larger model did not significantly outperform TextCNN.
   More data would likely benefit TextCRNN more.

# TEXTCRNN Model Training Loop

In [24]:
# ========= training the TextCRNN model ==========

# ======== Logging to wandb ===========

run3 = wandb.init(
    project = '23f3001514-t22026',
    entity  = '23f3001514-instituition',
    name    = 'textcrnn-cnn-lstm',
    config  = {
        'model'         : 'TextCRNN',
        'vocab_size'    : VOCAB_SIZE,
        'embedding_dim' : 128,
        'num_filters'   : 128,
        'hidden_size'   : 128,
        'lstm_layers'   : 2,
        'bidirectional' : True,
        'dropout'       : 0.5,
        'batch_size'    : 32,
        'learning_rate' : 0.001,
        'epochs'        : 20
    }
)

# =========== Loss function ===========

criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(crnn_model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='max', patience=3, factor=0.5)

best_val_accuracy2 = 0
patience_counter2  = 0

for epoch in range(NUM_EPOCHS):

    # ======== training phase =======
    crnn_model.train()
    train_correct2 = 0
    train_total2   = 0
    train_loss2    = 0

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer2.zero_grad()
        outputs = crnn_model(batch_x)
        loss    = criterion2(outputs, batch_y)
        loss.backward()
        nn.utils.clip_grad_norm_(crnn_model.parameters(), max_norm=1.0)
        optimizer2.step()

        train_loss2    += loss.item()
        preds           = outputs.argmax(dim=1)
        train_correct2 += (preds == batch_y).sum().item()
        train_total2   += batch_y.size(0)

    train_acc2 = train_correct2 / train_total2

    # ======== validation phase ==========
    
    crnn_model.eval()
    val_correct2   = 0
    val_total2     = 0
    val_preds_list = []
    val_true_list  = []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x  = batch_x.to(device)
            batch_y  = batch_y.to(device)
            outputs  = crnn_model(batch_x)
            preds    = outputs.argmax(dim=1)
            val_correct2 += (preds == batch_y).sum().item()
            val_total2   += batch_y.size(0)
            val_preds_list.extend(preds.cpu().numpy().tolist())
            val_true_list.extend(batch_y.cpu().numpy().tolist())

    val_acc2 = val_correct2 / val_total2
    val_f12  = f1_score(val_true_list, val_preds_list, average='macro')

    scheduler2.step(val_acc2)

    wandb.log({
        'epoch'        : epoch + 1,
        'train_loss'   : train_loss2 / len(train_loader),
        'train_acc'    : train_acc2,
        'val_acc'      : val_acc2,
        'val_macro_f1' : val_f12
    })

    if val_acc2 > best_val_accuracy2:
        best_val_accuracy2 = val_acc2
        torch.save(crnn_model.state_dict(), '/kaggle/working/best_crnn.pt')
        patience_counter2 = 0
    else:
        patience_counter2 += 1

    if patience_counter2 >= PATIENCE:
        print(f'early stopping at epoch {epoch + 1}')
        break

    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1}/{NUM_EPOCHS} | train acc: {train_acc2:.4f} | val acc: {val_acc2:.4f} | val f1: {val_f12:.4f}')

wandb.log({'best_val_acc': best_val_accuracy2})
run3.finish()

crnn_model.load_state_dict(torch.load('/kaggle/working/best_crnn.pt'))
print(f'TextCRNN training done! Best val accuracy: {best_val_accuracy2:.4f}')

# ============ getting test predictions ===============

crnn_model.eval()
crnn_test_probs_list = []

with torch.no_grad():
    for batch in test_loader:
        batch_x  = batch[0].to(device)
        outputs  = crnn_model(batch_x)
        probs    = torch.softmax(outputs, dim=1)
        crnn_test_probs_list.append(probs.cpu().numpy())

crnn_test_probs = np.concatenate(crnn_test_probs_list, axis=0)
print('CRNN test predictions shape:', crnn_test_probs.shape)

epoch 5/20 | train acc: 0.7781 | val acc: 0.7850 | val f1: 0.7395
epoch 10/20 | train acc: 0.9925 | val acc: 0.9825 | val f1: 0.9833
epoch 15/20 | train acc: 0.9931 | val acc: 0.9900 | val f1: 0.9898
early stopping at epoch 16


best_val_acc,▁
epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
train_acc,▁▂▃▅▆▆▇▇████████
train_loss,██▇▅▃▃▂▂▁▁▁▁▁▁▁▁
val_acc,▁▁▄▅▆▆▇█████████
val_macro_f1,▁▁▄▅▆▆▆█████████
best_val_acc,0.995
epoch,16
train_acc,0.99813
train_loss,0.00887
val_acc,0.9925


TextCRNN training done! Best val accuracy: 0.9950
CRNN test predictions shape: (500, 5)
